# Statistical analysis of attrition

In [1]:
# data manipulation libraries
import numpy as np
import pandas as pd

# data visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# statistical modules
from scipy.stats import chi2_contingency

In [2]:
# loading the dataset into a dataframe
attrition = pd.read_csv('attrition_df.csv')
print(f"The data has {attrition.shape[0]} rows and {attrition.shape[1]} columns.")
print(sorted(attrition.columns))
print("----")
attrition.sample(5)

The data has 2000000 rows and 21 columns.
['age', 'birth_date', 'department', 'dpt_acronym', 'ee#', 'employment_province', 'ethnicity', 'first_name', 'full_name', 'gender', 'hire_source', 'last_name', 'level', 'position', 'position_type', 'province_acronym', 'start_date', 'tenure', 'term_date', 'term_reason', 'work_location']
----


,ee#,first_name,last_name,full_name,position,position_type,start_date,term_date,tenure,birth_date,...,department,dpt_acronym,work_location,employment_province,province_acronym,hire_source,level,gender,ethnicity,term_reason
1286761,11286761,Steven,Holland,"Holland, Steven",Operations Analyst,Full Time - Regular,2001-01-30,2011-12-25,10.9,1943-06-29,...,Service Delivery & Operations,SDO,Longueuil,Quebec,QC,direct,Professional,Man,African descent,dismissal
508802,10508802,Lisa,Berry,"Berry, Lisa",IT Director,Full Time - Regular,2021-11-16,NaN,3.9,1992-10-30,...,Technology Solutions & Services,TSS,Mississauga,Ontario,ON,referral,Director,Man,Caucasian,NaN
603205,10603205,Barry,Sullivan,"Sullivan, Barry",IT Support Analyst,Full Time - Regular,2021-11-06,NaN,3.9,1996-07-30,...,Technology Solutions & Services,TSS,Laval,Quebec,QC,direct,Professional,Man,Caucasian,NaN
891739,10891739,Rose,Moran,"Moran, Rose",Corporate Communications Specialist,Full Time - Regular,1963-07-18,1994-06-07,30.9,1932-05-07,...,Communications & Brand Strategy,CBS,Ottawa,Ontario,ON,referral,Professional,Woman,Bi-racial,other
1028755,11028755,Hannah,Greene,"Greene, Hannah",Communications Manager,Full Time - Regular,1959-12-17,1989-11-17,29.9,1927-02-13,...,Communications & Brand Strategy,CBS,Burnaby,British Columbia,BC,agency,Professional,Man,Caucasian,resignation


In [3]:
# dropping columns that aren't interesting for predictors
attrition.drop(columns= ['birth_date', 'dpt_acronym', 'first_name', 'full_name', 
                         'last_name', 'position', 'province_acronym',
                         'start_date', 'term_date'], 
               inplace= True)

attrition.sample(5)

,ee#,position_type,tenure,age,department,work_location,employment_province,hire_source,level,gender,ethnicity,term_reason
84842,10084842,Full Time - Regular,4.6,58.4,Client Experience & Engagement,Toronto,Ontario,direct,Professional,Man,Caucasian,other
221047,10221047,Full Time - Regular,33.0,60.0,Service Delivery & Operations,Hamilton,Ontario,direct,Professional,Man,Caucasian,other
964052,10964052,Full Time - Regular,2.0,46.4,Technology Solutions & Services,Toronto,Ontario,direct,Professional,Woman,Pacific Islander,NaN
939708,10939708,Full Time - Regular,20.7,67.0,Human Capital Strategy,St. John’s,Newfoundland and Labrador,agency,Director,Man,Caucasian,NaN
526598,10526598,Full Time - Regular,22.5,51.1,Technology Solutions & Services,Ottawa,Ontario,agency,Professional,Man,Caucasian,NaN


In [4]:
# getting basic information about the data
summary = []

for i in attrition.columns:
    column_info = {}
    column_info['name'] = attrition[i].name
    column_info['data type'] = attrition[i].dtypes
    column_info['example'] = attrition[i].iloc[0]
    column_info['unique#'] = attrition[i].nunique()
    summary.append(column_info)

summary = pd.DataFrame(summary)
summary

,name,data type,example,unique#
0,ee#,int64,10000000,2000000
1,position_type,object,Full Time - Temporary,4
2,tenure,float64,11.1,411
3,age,float64,36.7,461
4,department,object,Regulatory Compliance & Risk Management,9
5,work_location,object,Vancouver,25
6,employment_province,object,British Columbia,13
7,hire_source,object,direct,3
8,level,object,Director,4
9,gender,object,Man,3


## Dataframe preparation for statistical analysis

To identify whether resignations happen more frequently in specific departments, work locations or other data points, a good measure to take generally for machine learning first but can also be useful in statistics is to transform the dataframe's variables into binary dummy variables.

For the sake of demonstration, let's focus on the work location column as predictor and resignations as target.

In [5]:
# isolate 'term_reasons' and 'work_location' columns
attrition_dummy = attrition[['work_location', 'term_reason']]

# break down the dataframe into binary variables & keeping only resignations as target
attrition_dummy = pd.get_dummies(data= attrition_dummy,
                                 dtype= 'float64')
attrition_dummy.drop(columns= ['term_reason_dismissal', 'term_reason_end of contract', 
                               'term_reason_other', 'term_reason_without cause'],
                     inplace= True)

print(sorted(attrition_dummy.columns))
attrition_dummy.sample(5)

['term_reason_resignation', 'work_location_Brampton', 'work_location_Burnaby', 'work_location_Calgary', 'work_location_Charlottetown', 'work_location_Edmonton', 'work_location_Fredericton', 'work_location_Gatineau', 'work_location_Halifax', 'work_location_Hamilton', 'work_location_Iqaluit', 'work_location_Laval', 'work_location_Longueuil', 'work_location_Mississauga', 'work_location_Montreal', 'work_location_Ottawa', 'work_location_Quebec City', 'work_location_Red Deer', 'work_location_Regina', 'work_location_St. John’s', 'work_location_Surrey', 'work_location_Toronto', 'work_location_Vancouver', 'work_location_Whitehorse', 'work_location_Winnipeg', 'work_location_Yellowknife']


,work_location_Brampton,work_location_Burnaby,work_location_Calgary,work_location_Charlottetown,work_location_Edmonton,work_location_Fredericton,work_location_Gatineau,work_location_Halifax,work_location_Hamilton,work_location_Iqaluit,...,work_location_Red Deer,work_location_Regina,work_location_St. John’s,work_location_Surrey,work_location_Toronto,work_location_Vancouver,work_location_Whitehorse,work_location_Winnipeg,work_location_Yellowknife,term_reason_resignation
15395,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
840589,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
160884,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1647832,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1374490,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# instantiate the work_locations as predictors
work_locations = [col for col in attrition_dummy.columns if col != 'term_reason_resignation']

# analyse each predictor against the target
correlations = []
for predictor in work_locations:
    table = pd.crosstab(attrition_dummy[predictor], attrition_dummy['term_reason_resignation'])
    chi2, p, _, _ = chi2_contingency(table)
    n = table.values.sum()
    phi = (chi2 / n) ** 0.5
    correlations.append({'predictor': predictor, 'chi2': chi2, 'phi': phi, 'p_value': p})

# display the results in a dataframe and only keep 5 highest correlations
correlations = pd.DataFrame(correlations)
correlations = correlations.nlargest(5, 'phi')
correlations

,predictor,chi2,phi,p_value
17,work_location_Regina,6879.215622,0.058648,0.000000e+00
20,work_location_Toronto,46.591594,0.004827,8.743453e-12
0,work_location_Brampton,26.419101,0.003634,2.748094e-07
19,work_location_Surrey,22.951685,0.003388,1.661245e-06
16,work_location_Red Deer,18.534171,0.003044,1.668854e-05


Work location 'Regina' has a weak but statistically significant association with resignations (small phi coefficient, but extremely high chi-squared and tiny p-value).